In [5]:
import pandas as pd
df = pd.read_csv('data_Histo/vnindex_eda_output.csv')
df.columns

Index(['symbol', 'date', 'high_price', 'low_price', 'open_price',
       'average_price', 'close_price', 'basic_price', 'adj_ratio', 'unit',
       'vol_total', 'vol_deal', 'vol_putth', 'val_total', 'val_putth',
       'buy_vol_foreign', 'buy_val_foreigh', 'sell_vol_foreign',
       'sel_val_foreign', 'buy_count', 'buy_vol', 'sell_count', 'sell_vol',
       'foreign_room', 'prop_trading_deal', 'prop_trading_putth',
       'prop_trading_net', 'year'],
      dtype='object')

In [6]:
df.head()

,symbol,date,high_price,low_price,open_price,average_price,close_price,basic_price,adj_ratio,unit,...,sel_val_foreign,buy_count,buy_vol,sell_count,sell_vol,foreign_room,prop_trading_deal,prop_trading_putth,prop_trading_net,year
0,VNINDEX,2010-01-04,517.05,501.74,501.74,517.05,517.05,494.77,1.0,1.0,...,1.863560e+11,35017.0,94121000.0,20679.0,47387200.0,0.0,NaN,NaN,NaN,2010
1,VNINDEX,2010-01-05,539.39,529.23,529.23,532.53,532.53,517.05,1.0,1.0,...,2.835830e+11,38919.0,114175000.0,34412.0,87391200.0,0.0,NaN,NaN,NaN,2010
2,VNINDEX,2010-01-06,538.84,526.37,529.47,534.46,534.46,532.53,1.0,1.0,...,1.578620e+11,44534.0,121988000.0,54391.0,113832000.0,0.0,NaN,NaN,NaN,2010
3,VNINDEX,2010-01-07,540.77,530.68,536.78,533.34,533.34,534.46,1.0,1.0,...,1.495040e+11,48977.0,127322000.0,46431.0,107237000.0,0.0,NaN,NaN,NaN,2010
4,VNINDEX,2010-01-08,544.49,520.90,540.95,520.90,520.90,533.34,1.0,1.0,...,1.995820e+11,47171.0,112076000.0,56106.0,139754000.0,0.0,NaN,NaN,NaN,2010


In [7]:
len(df)

3826

In [3]:
import pandas as pd
df = pd.read_parquet('data_News/equity_news_content_sentiment_ratios.parquet')
df.head(1)

,link,publication_date,domain_norm,category,title,description,keywords_norm,content,Tokenize_content_sentences,Tokenize_content,...,pos_term_count,neg_term_count,neutral_term_count,pos_ratio,neg_ratio,neutral_ratio,sentiment_coverage_ratio,polarity_count,sentiment_score,sentiment_label
0,https://vietstock.vn/2010/01/co-dong-noi-bo-ag...,2010-01-01 19:23:00,vietstock.vn,Giao dịch nội bộ,Cổ đông nội bộ AGF và BVH vi phạm CBTT,(Vietstock) - Sở GDCK TPHCM (HOSE) thông báo v...,<NA>,"Cụ thể, bà Nguyễn Thị Kim Lan đã mua 884,140 c...","[[cụ_thể, bà, nguyễn_thị_kim_lan, đã, mua, cổ_...","[cụ_thể, bà, nguyễn_thị_kim_lan, đã, mua, cổ_p...",...,2,0,8,0.060606,0.0,0.181818,0.242424,4,1.0,positive


In [11]:
len(df)


126576

In [4]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data_News/equity_news_content_sentiment_ratios.parquet")
OUTPUT_PATH = Path("data_News/label_studio_ground_truth_stratified_200.csv")

RANDOM_STATE = 42

sample_plan = {
    "positive": 60,
    "negative": 60,
    "neutral": 60,
}

df = pd.read_parquet(INPUT_PATH).reset_index(names="source_row_id")

df["sentiment_label"] = (
    df["sentiment_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df = df.loc[
    df["content"].notna()
    & df["content"].astype(str).str.strip().ne("")
    & df["sentiment_label"].isin(sample_plan.keys())
].copy()

samples = []

for label, n in sample_plan.items():
    label_df = df.loc[df["sentiment_label"].eq(label)].copy()

    if len(label_df) < n:
        raise ValueError(f"Not enough rows for {label}: need {n}, got {len(label_df)}")

    samples.append(
        label_df.sample(n=n, random_state=RANDOM_STATE)
    )

sample_df = (
    pd.concat(samples, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

keep_columns = [
    "source_row_id",
    "publication_date",
    "domain_norm",
    "category",
    "title",
    "description",
    "content",
    "sentiment_score",
    "sentiment_label",
]

keep_columns = [col for col in keep_columns if col in sample_df.columns]
sample_df = sample_df[keep_columns].copy()

sample_df = sample_df.rename(columns={
    "sentiment_label": "model_sentiment_label",
    "sentiment_score": "model_sentiment_score",
})

sample_df.insert(0, "annotation_id", range(1, len(sample_df) + 1))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
print("Rows:", len(sample_df))
print(sample_df["model_sentiment_label"].value_counts())

Saved: data_News\label_studio_ground_truth_stratified_200.csv
Rows: 180
model_sentiment_label
positive    60
neutral     60
negative    60
Name: count, dtype: Int64


In [1]:
import re
import pandas as pd
import pyarrow.parquet as pq

DATA_PATH = "../data_news/equity_news_tokenized_underthesea.parquet"

df = pq.read_table(DATA_PATH, columns=["title", "content", "publication_date", "link"]).to_pandas()
df["content"] = df["content"].fillna("")
print("Tổng số bài:", len(df))


Tổng số bài: 126576


In [2]:
def find_word_in_articles(df, word, n_samples=10, window=60, seed=42):
    search_phrase = word.replace("_", " ")
    pattern = re.compile(r"\b" + re.escape(search_phrase) + r"\b", flags=re.IGNORECASE)

    mask = df["content"].str.contains(pattern, regex=True)
    matched = df.loc[mask]

    print(f"Từ khoá: '{word}'  (tìm dạng: '{search_phrase}')")
    print(f"Số bài chứa từ này: {len(matched)} / {len(df)}")

    if matched.empty:
        return matched

    sample = matched.sample(min(n_samples, len(matched)), random_state=seed)
    for _, row in sample.iterrows():
        m = pattern.search(row["content"])
        start = max(0, m.start() - window)
        end = min(len(row["content"]), m.end() + window)
        snippet = row["content"][start:end].replace("\n", " ")
        print("-" * 80)
        print("Tiêu đề:", row["title"])
        print("Ngày:", row["publication_date"])
        print("...", snippet, "...")

    return matched


In [6]:
matched = find_word_in_articles(df, "gian_lận", n_samples=10)


Từ khoá: 'gian_lận'  (tìm dạng: 'gian lận')
Số bài chứa từ này: 296 / 126576
--------------------------------------------------------------------------------
Tiêu đề: Việc các doanh nghiệp quay lại thị trường nội địa là tất yếu
Ngày: 2023-04-30 04:00:00
... uản lý nhà nước trên thị trường, kiểm tra chống buôn lậu và gian lận thương mại, sản xuất và kinh doanh hàng giả, ngăn ngừa kịp  ...
--------------------------------------------------------------------------------
Tiêu đề: Nhận diện được các gian lận trong báo cáo tài chính
Ngày: 2019-10-04 11:00:00
... thiệu khá nhiều mô hình định lượng giúp phát hiện chính xác gian lận của các doanh nghiệp. Các mô hình đó sẽ được các chuyên gia ...
--------------------------------------------------------------------------------
Tiêu đề: Chính sách TTCK 2017: Điểm cộng từ chứng khoán phái sinh
Ngày: 2017-12-29 13:02:00
... ay thì quy định mới chỉ còn áp dụng cho “hành vi trốn thuế, gian lận thuế” hay “không chấp hành”. Lấy ý kiến Dự thảo đề nghị xâ

In [32]:
import pandas as pd
df = pd.read_csv('../News/Build_sentiment_label/Lexicon_based/data/candidate_ngram_terms_dictionary_pmi.csv')
len(df)
df[df["sentiment_label"] == "neutral"].head(50)

,term,ngram_n,tf,df,df_ratio,avg_tf_per_doc,candidate_score,pos_pmi_mean,neg_pmi_mean,pos_seed_matches,neg_seed_matches,so_score,so_score_z,sentiment_label,label_source,pmi_confidence
4,triệu đồng,2,20758,11364,0.089780,1.826646,50033.263997,0.275024,0.639399,46.0,57.0,-0.364375,0.380695,neutral,pmi,reliable
21,tỉ đồng,2,8155,2028,0.016022,4.021203,33707.124765,1.202946,1.646155,46.0,56.0,-0.443209,0.316078,neutral,pmi,reliable
35,công_bố thông_tin,2,8837,4815,0.038040,1.835306,28887.332137,0.333378,0.722138,46.0,57.0,-0.388760,0.360708,neutral,pmi,reliable
49,tỷ_lệ số_lượng,2,8857,7911,0.062500,1.119580,24555.768777,-5.759303,-5.471349,30.0,26.0,-0.287954,0.443336,neutral,pmi,reliable
51,phiên phiên,2,5787,1971,0.015572,2.936073,24084.351429,-3.608765,-3.376003,34.0,38.0,-0.232762,0.488575,neutral,pmi,reliable
55,ủy_viên hđqt,2,7054,4547,0.035923,1.551353,23462.755514,-2.498380,-2.219831,42.0,49.0,-0.278549,0.451044,neutral,pmi,reliable
59,chi_phí tài_chính,2,7610,6277,0.049591,1.212363,22858.913131,0.140994,0.511901,46.0,57.0,-0.370907,0.375342,neutral,pmi,reliable
66,đầu_tư chứng_khoán,2,7200,5918,0.046755,1.216627,22051.319781,0.562978,0.939340,46.0,57.0,-0.376361,0.370871,neutral,pmi,reliable
69,lãi vay,2,6694,4871,0.038483,1.374256,21804.676053,0.695161,1.102442,45.0,57.0,-0.407282,0.345526,neutral,pmi,reliable
73,phát_hành trái_phiếu,2,5541,2586,0.020430,2.142691,21556.439963,0.868104,1.131716,46.0,57.0,-0.263612,0.463288,neutral,pmi,reliable


In [27]:
df1 = df[(df['df'] == 1)]
len(df1)
df1.to_csv('../News/Build_sentiment_label/Lexicon_based/data/candidate_ngram_terms_filtered.csv', index=False)
